# 01 — Quickstart

Load the recommended HOMER π, query a mouse region, see its top-K human partners with their multi-source trust tier.

This notebook does **not** re-fit the model — it loads pre-computed outputs from `outputs/coupling/`. To regenerate, run `experiments/anchor_packs/compose_all.py` then this notebook again.

**Sections:**
1. Setup — load π, atlases, trust map
2. Single-region query (interactive)
3. Compare two π files (production vs production+packs)
4. Bulk region translation
5. 3D brain visualisation

In [1]:
# Setup
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from homer.data import load_cached

ROOT = Path.cwd().parent
ANN = ROOT / 'outputs' / 'anndata'
COUP = ROOT / 'outputs' / 'coupling'

# Load atlases (instant — h5ad caches)
M, _ = load_cached('mouse', cache_dir=ANN)
H, _ = load_cached('human', cache_dir=ANN)
print(f'Mouse parcels: {len(M.var)}, Human parcels: {len(H.var)}')

# Load the two production π
pi_strict = np.load(COUP / 'pi_fc_plus_SC.npy')
pi_packed = np.load(COUP / 'pi_fc_plus_SC_with_all_packs.npy')
print(f'π strict (Garin anchors only):     {pi_strict.shape}')
print(f'π packed (+ 5 default anchor packs): {pi_packed.shape}')

# Load multi-source trust map for the packed π
trust = np.load(COUP / 'trust_multisource_all_packs.npz', allow_pickle=True)
trust_tier = trust['evidence_tier']
print(f'\nTrust tiers (over {len(trust_tier)} mouse parcels):')
for t in ['anchored_and_validated', 'anchored_only', 'validated_only', 'structural', 'low_evidence']:
    n = int((trust_tier == t).sum())
    print(f'  {t:25s} {n:4d}  ({n/len(trust_tier):>6.1%})')

Mouse parcels: 1864, Human parcels: 2094
π strict (Garin anchors only):     (1864, 2094)
π packed (+ 5 default anchor packs): (1864, 2094)

Trust tiers (over 1864 mouse parcels):
  anchored_and_validated     354  ( 19.0%)
  anchored_only               65  (  3.5%)
  validated_only             665  ( 35.7%)
  structural                 233  ( 12.5%)
  low_evidence               547  ( 29.3%)


## 2. Query a single mouse region — interactive

Pick a mouse region from the dropdown. Adjust top-K with the slider. The display shows:
- the top-K human partners with their MNI coords and region names
- the predicted region's trust tier (multi-source evidence label)
- the row's mass concentration (sharpness of the prediction)

In [2]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Build a sorted list of unique mouse regions (most-represented first)
region_counts = M.var['region'].value_counts()
mouse_regions = region_counts.index.tolist()

# Widgets
region_dropdown = widgets.Dropdown(
    options=mouse_regions, value=mouse_regions[0],
    description='Mouse region:', layout=widgets.Layout(width='500px'),
    style={'description_width': '120px'},
)
parcel_dropdown = widgets.Dropdown(
    options=[], description='Mouse parcel:',
    layout=widgets.Layout(width='500px'),
    style={'description_width': '120px'},
)
topk_slider = widgets.IntSlider(value=5, min=1, max=10, step=1,
                                  description='top-K:',
                                  style={'description_width': '120px'})
pi_choice = widgets.Dropdown(options=[('production + all packs', 'packed'),
                                        ('production point-anchor only', 'strict')],
                              value='packed', description='Which π:',
                              style={'description_width': '120px'},
                              layout=widgets.Layout(width='500px'))
out = widgets.Output()

def update_parcels(*_):
    region = region_dropdown.value
    parcels = M.var[M.var['region'] == region].index.tolist()
    parcel_dropdown.options = parcels
    parcel_dropdown.value = parcels[0] if parcels else None

def render(*_):
    with out:
        clear_output()
        parcel_id = parcel_dropdown.value
        if parcel_id is None:
            print('No parcel selected.')
            return
        pi = pi_packed if pi_choice.value == 'packed' else pi_strict
        # Find the row index for this parcel
        row = M.var.index.get_loc(parcel_id)
        k = topk_slider.value
        topk = pi[row].argsort()[::-1][:k]
        scores = pi[row][topk]

        tier = trust_tier[row] if pi_choice.value == 'packed' else 'n/a (strict π)'
        concentration = pi[row].max() / pi[row].sum()

        print(f'Mouse parcel: {parcel_id} ({M.var.iloc[row]["region"]})')
        print(f'  xyz: ({M.var.iloc[row].x:+.2f}, {M.var.iloc[row].y:+.2f}, {M.var.iloc[row].z:+.2f}) mm')
        print(f'  Trust tier:        {tier}')
        print(f'  Row concentration: {concentration:.1%} (peak / row sum)')
        print(f'\nTop {k} human partners ({pi_choice.value} π):')
        rows = []
        for r in topk:
            rows.append({
                'human_parcel': H.var.index[r],
                'region': H.var.iloc[r]['region'],
                'x':  f'{H.var.iloc[r].x:+.1f}',
                'y':  f'{H.var.iloc[r].y:+.1f}',
                'z':  f'{H.var.iloc[r].z:+.1f}',
                'pi':       f'{pi[row, r]:.4f}',
                'pi_frac':  f'{pi[row, r] / pi[row].sum():.1%}',
            })
        display(pd.DataFrame(rows))

region_dropdown.observe(update_parcels, names='value')
update_parcels()
for w in (region_dropdown, parcel_dropdown, topk_slider, pi_choice):
    w.observe(render, names='value')

display(widgets.VBox([region_dropdown, parcel_dropdown, topk_slider, pi_choice, out]))
render()

## 3. Compare predictions between strict π and packed π

For the selected parcel above, run the cell below to see how strict (Garin-only) and packed (with 5 anchor packs) predictions differ side-by-side. This makes the effect of anchor packs visible per-parcel.

In [3]:
def side_by_side(parcel_id, k=5):
    row = M.var.index.get_loc(parcel_id)
    top_strict = pi_strict[row].argsort()[::-1][:k]
    top_packed = pi_packed[row].argsort()[::-1][:k]
    table = []
    for i in range(k):
        rs, rp = top_strict[i], top_packed[i]
        table.append({
            'rank': i + 1,
            'strict region':    H.var.iloc[rs]['region'][:30],
            'strict π':          f'{pi_strict[row, rs]:.4f}',
            'packed region':    H.var.iloc[rp]['region'][:30],
            'packed π':          f'{pi_packed[row, rp]:.4f}',
        })
    return pd.DataFrame(table)

# Example: see how Motor parcel mapping differs
motor_parcel = M.var[M.var['region'].str.contains('Motor', case=False, na=False)].index[0]
print(f'Mouse parcel {motor_parcel} ({M.var.loc[motor_parcel, "region"]}):')
side_by_side(motor_parcel, k=5)

Mouse parcel 3 (L_Motor and premotor):


,rank,strict region,strict π,packed region,packed π
0,1,L_Motor and premotor,0.0005,L_1040,0.0002
1,2,R_1047,0.0000,L_1002,0.0002
2,3,R_348,0.0000,L_1004,0.0001
3,4,R_351,0.0000,L_946,0.0000
4,5,L_351,0.0000,R_1002,0.0000


## 4. Bulk region translation

For a whole mouse region, aggregate π across all its member parcels and show the top human regions.

In [4]:
def translate_region(region_query, pi=pi_packed, top_k=5):
    """Aggregate π over mouse parcels matching region_query, return top-K human regions."""
    mask = M.var['region'].str.contains(region_query, case=False, na=False)
    if mask.sum() == 0:
        return f'No mouse parcels match {region_query!r}'
    pi_M = pi[mask].sum(axis=0)
    pi_M /= pi_M.sum()
    # Aggregate by human region name
    h_regions = H.var['region'].copy()
    agg = pd.Series(pi_M).groupby(h_regions.values).sum().sort_values(ascending=False).head(top_k)
    out = pd.DataFrame({
        'human_region': agg.index,
        'mass_share':   [f'{v:.1%}' for v in agg.values],
    })
    print(f'Mouse "{region_query}" — {int(mask.sum())} parcels — top-{top_k} human partners:')
    return out

translate_region('Hippocampal', top_k=10)

"No mouse parcels match 'Hippocampal'"

Try other regions:

In [5]:
# Visualise where Motor maps to
print(translate_region('Motor', top_k=8))
print('---')
print(translate_region('Amygdala', top_k=8))
print('---')
print(translate_region('Thalamus', top_k=8))

Mouse "Motor" — 2 parcels — top-8 human partners:
  human_region mass_share
0       L_1002      23.8%
1       L_1040      23.5%
2       R_1002      18.1%
3       R_1040      14.3%
4       L_1004       8.2%
5       R_1004       4.8%
6        L_946       2.2%
7        R_997       1.8%
---
Mouse "Amygdala" — 2 parcels — top-8 human partners:
         human_region mass_share
0               L_145      73.9%
1               R_145      26.1%
2  L_Olfactory cortex       0.1%
3  R_Olfactory cortex       0.0%
4               R_151       0.0%
5               L_151       0.0%
6          L_Amygdala       0.0%
7               L_371       0.0%
---
Mouse "Thalamus" — 4 parcels — top-8 human partners:
     human_region mass_share
0      L_Thalamus      25.0%
1      R_Thalamus      25.0%
2  R_Hypothalamus      25.0%
3  L_Hypothalamus      25.0%
4        L_Tectum       0.0%
5           L_226       0.0%
6        R_Tectum       0.0%
7           L_257       0.0%


## 5. 3D brain view

Render the mouse brain coloured by Garin functional network. Each dot is a parcel.

In [6]:
from homer.viz.notebook import plot_brain_3d
plot_brain_3d(M, color_by='network', title='Mouse atlas — network coloring')

## What's next

- See `notebooks/02_trust_map.ipynb` for the per-parcel evidence-tier exploration.
- See `notebooks/03_anchor_packs.ipynb` to compare the impact of each anchor pack.
- See `notebooks/04_methodology.ipynb` for the FGW solver step-by-step.
- See `docs/04_anchor_packs.md` for how to add new packs.

For programmatic use, see the snippet in `README.md`.